In [1]:
import os, sys, pandas, numpy, re
print(sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
print("Spark version:", spark.version)
#spark.range(5).show()

C:\Users\suraj\AppData\Local\pypoetry\Cache\virtualenvs\data-transformations-prwh0KIk-py3.11\Scripts\python.exe
JAVA_HOME: C:\JDK11
HADOOP_HOME: C:\Users\suraj\Tools\winutils\hadoop-3.3.5
Spark version: 3.5.1


In [2]:
from datetime import datetime, date
import calendar

def check_date_in_range(api_response: dict, date_to_check: str) -> bool:
    """
    Returns True if the given date_to_check (YYYY-MM-DD)
    lies between both 'clicks' and 'visits' date ranges.
    """

    def month_start_end(month_str: str):
        """Convert YYYY-MM to (start_date, end_date)."""
        year, month = map(int, month_str.split('-'))
        start = date(year, month, 1)
        last_day = calendar.monthrange(year, month)[1]
        end = date(year, month, last_day)
        return start, end

    # Extract start and end for clicks
    clicks_start, clicks_end = month_start_end(api_response["clicks"]["start_month"])
    clicks_end = month_start_end(api_response["clicks"]["end_month"])[1]

    # Extract start and end for visits
    visits_start, visits_end = month_start_end(api_response["visits"]["start_month"])
    visits_end = month_start_end(api_response["visits"]["end_month"])[1]

    # Compute global min and max across both
    global_start = max(clicks_start, visits_start)
    global_end = min(clicks_end, visits_end)

    # Convert user date
    check_date = datetime.strptime(date_to_check, "%Y-%m-%d").date()

    # Logic: check within both clicks and visits AND within global bounds
    within_clicks = clicks_start <= check_date <= clicks_end
    within_visits = visits_start <= check_date <= visits_end
    within_global = global_start <= check_date <= global_end

    return within_clicks and within_visits and within_global


# Example usage
api_data = {
    "clicks": {"start_month": "2023-01", "end_month": "2023-04"},
    "visits": {"start_month": "2023-01", "end_month": "2023-03"},
}

print(check_date_in_range(api_data, "2023-03-15"))  # ✅ True
print(check_date_in_range(api_data, "2023-04-20"))  # ❌ False


True
False
